# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. All dataset entities are referenced by their Croissant `@id` fields for compatibility and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant pandas matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The metadata object gives a high-level overview of the dataset and its schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

**Note:** Entities are referenced by their unique Croissant `@id` for accurate and reproducible access.

In [ ]:
# List all available record sets and their fields, referencing by @id
record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Fields/Columns:")
    for f in rs.fields:
        print(f"    - Field @id: {f.id}, name: {f.name}, type: {f.data_type}")
    print()

### Displaying Example Records
Let's display the first example record of each record set using its Croissant `@id`.

In [ ]:
# Show the first example record from each record set (by @id)
for rs in record_sets:
    print(f"=== RecordSet: {rs.name} (@id: {rs.id}) ===")
    gen = dataset.records(record_set=rs.id)
    try:
        first = next(gen)
        print(first)
    except Exception as e:
        print("No records found or failed to load.")
    print()

## 3. Data Extraction
Load all records from the main data record set(s) into DataFrames for analysis.

We use the list of record set `@id`s obtained above.

_If you are exploring in a new notebook, please refer back to the list above and update accordingly._

In [ ]:
# Collect all record set @id's for loading
record_set_ids = [rs.id for rs in metadata.record_sets]

# Load each record set into a pandas DataFrame (referenced by record set @id)
dataframes = {}
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded {len(df)} records from RecordSet @id: {rec_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records loaded for RecordSet @id: {rec_id}")

## 4. Exploratory Data Analysis (EDA)
Choose key numeric and grouping fields for analysis. Reference every column by its Croissant `@id` (these can be found in each record set's field listing above).

We'll demonstrate filtering, normalization, and grouping. Adjust the field `@id`s as necessary for your dataset.

In [ ]:
# Example: Select one record set for analysis
# (Replace with the relevant record set @id and numeric/group fields for your dataset)
if len(dataframes) > 0:
    # Pick the first non-empty record set
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using RecordSet @id: {record_set_id}")
    
    # Detect numeric columns (float or int types)
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    print("Numeric field candidates:", numeric_fields)
    
    # Choose first numeric field for demonstration
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Analyzing numeric field (by column name, which equals @id): {numeric_field}")
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized values for {numeric_field}:")
        display(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())
        # Grouping if possible
        group_candidates = [c for c in df.columns if c not in numeric_fields]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by field (by column name/@id): {group_field}")
            grouped = filtered_df.groupby(group_field).mean(numeric_only=True)
            display(grouped.head())
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize numeric distributions and relationships using field `@id`s. Please adjust the field `@id` variables as appropriate for your analysis.

In [ ]:
# Example: Visualizing the distribution of a numeric field
if len(dataframes) > 0 and numeric_fields:
    plt.figure(figsize=(8,5))
    df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field}')
    plt.show()
    # If grouped field exists, plot average numeric value per group
    if group_candidates:
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
        group_means.plot(kind='bar', figsize=(10,4), title=f'Mean {numeric_field} by {group_field}')
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
We have explored the FAIR² dataset using the `mlcroissant` library, loading record sets by their Croissant `@id`, and performed basic data analysis and visualization. See the above sections for more comprehensive schema-driven workflows, and adapt field and record set `@id`s as new releases or record sets become available.